# PolyAI Training on Google Colab (Tesla T4)

This notebook sets up and runs your Rust-based MCTS Zero training on Colab's GPU.

## 1. Check GPU

In [2]:
!nvidia-smi

Tue Feb  3 22:32:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Rust

In [3]:
# Install Rust
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ['PATH']}"
!rustc --version

info: downloading installer
info: profile set to 'default'
info: default host triple is x86_64-unknown-linux-gnu
info: syncing channel updates for 'stable-x86_64-unknown-linux-gnu'
info: latest update on 2026-01-22, rust version 1.93.0 (254b59607 2026-01-19)
info: downloading component 'cargo'
info: downloading component 'clippy'
info: downloading component 'rust-docs'
info: downloading component 'rust-std'
info: downloading component 'rustc'
info: downloading component 'rustfmt'
info: installing component 'cargo'
 10.3 MiB /  10.3 MiB (100 %)   8.7 MiB/s in  1s         
info: installing component 'clippy'
info: installing component 'rust-docs'
 20.7 MiB /  20.7 MiB (100 %)   4.8 MiB/s in  4s         
info: installing component 'rust-std'
 28.2 MiB /  28.2 MiB (100 %)  10.9 MiB/s in  3s         
info: installing component 'rustc'
 74.4 MiB /  74.4 MiB (100 %)  10.5 MiB/s in  7s         
info: installing component 'rustfmt'
info: default toolchain set to 'stable-x86_64-unknown-linux-gnu

## 3. Fetch Repo

In [ ]:
# !git clone https://github.com/HenBOMB/Polyfish
# %cd Polyfish/polyfish-rs
!git fetch
!git pull

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 3.25 KiB | 555.00 KiB/s, done.
From https://github.com/HenBOMB/Polyfish
   8373d78..ac7079a  main       -> origin/main
Updating 8373d78..ac7079a
Fast-forward
 polyfish-rs/Cargo.lock          | 503 ++++++++++++++++++++--------------------
 polyfish-rs/Cargo.toml          |   9 +-
 polyfish-rs/main.ipynb          |  42 ++++
 polyfish-rs/setup_cuda_colab.sh |  52 -----
 polyfish-rs/training_log.csv    |   1 -
 5 files changed, 294 insertions(+), 313 deletions(-)
 delete mode 100755 polyfish-rs/setup_cuda_colab.sh
 delete mode 100644 polyfish-rs/training_log.csv


In [11]:
!sh ./check_readiness.sh

=== Production Readiness Check ===

1. Checking CUDA availability...
/opt/bin/nvidia-smi
Tesla T4, 550.54.15, 15360 MiB
✅ CUDA detected

2. Checking directory structure...
✅ Directories created

3. Checking Rust build...
   Compiling candle-core v0.8.4
   Compiling candle-nn v0.8.4
   Compiling polyfish v0.1.0 (/content/Polyfish/polyfish-rs)
    Finished `release` profile [optimized] target(s) in 1m 29s
✅ Rust build successful

4. Checking Python dependencies...
PyTorch: 2.9.0+cu126
SafeTensors: OK
✅ Python dependencies OK

5. Running test game...
Using device: Cpu
Starting with new random model.
Starting parallel self-play: 1 games with 10 MCTS iterations
Seed: 1770160163 -> Step: 46 -> 60 -> Research Hunting -> End Turn -> Step: 157 -> 142 -> Research Fishing -> End Turn -> Step: 60 -> 73 -> Harvest Fruit at 61 -> End Turn -> Harvest Fruit at 158 -> Step: 142 -> 127 -> End Turn -> Step: 73 -> 58 -> Train Warrior at 46 -> End Turn -> Step: 127 -> 140 -> Harvest Fruit at 172 -> Choose 

## 4. Build with CUDA Support

In [12]:
# Build in release mode with CUDA
!cargo build --release --features cuda --bin self_play

   Compiling polyfish v0.1.0 (/content/Polyfish/polyfish-rs)
    Finished `release` profile [optimized] target(s) in 53.81s


In [21]:
# Quick test to verify GPU works
!cargo run --release --features cuda --bin benchmark

    Updating crates.io index
  Downloaded candle-ug v0.9.2
  Downloaded float8 v0.6.1
  Downloaded pulp-wasm-simd-flag v0.1.0
  Downloaded gemm-f32 v0.19.0
  Downloaded gemm-c32 v0.19.0
  Downloaded gemm-f64 v0.19.0
  Downloaded gemm-f16 v0.19.0
  Downloaded gemm-c64 v0.19.0
  Downloaded allocator-api2 v0.2.21
  Downloaded yoke-derive v0.8.1
  Downloaded thiserror v2.0.18
  Downloaded thiserror-impl v2.0.18
  Downloaded ug v0.5.0
  Downloaded candle-nn v0.9.2
  Downloaded safetensors v0.7.0
  Downloaded typed-path v0.12.2
  Downloaded gemm-common v0.19.0
  Downloaded zip v7.2.0
  Downloaded pulp v0.22.2
  Downloaded yoke v0.8.1
  Downloaded ug-cuda v0.5.0
  Downloaded candle-core v0.9.2
  Downloaded gemm v0.19.0
  Downloaded foldhash v0.2.0
  Downloaded float8 v0.3.0
  Downloaded candle-kernels v0.9.2
  Downloaded cudarc v0.19.0
  Downloaded cudarc v0.17.8
   Compiling proc-macro2 v1.0.106
   Compiling unicode-ident v1.0.22
   Compiling quote v1.0.44
   Compiling libc v0.2.180
   Compi

In [17]:
!sh ./setup_cuda_colab.sh

=== Configuring CUDA Build for T4 GPU ===

1. CUDA Version:
Cuda compilation tools, release 12.5, V12.5.82
7.5

2. Setting build environment...
✅ Environment configured for CUDA 12.4 + T4

3. Cleaning previous builds...
     Removed 1629 files, 565.2MiB total
✅ Clean complete

4. Building with CUDA support...
   This will take ~5-10 minutes on first build...
    Updating git repository `https://github.com/huggingface/candle.git`
error: failed to get `candle-core` as a dependency of package `polyfish v0.1.0 (/content/Polyfish/polyfish-rs)`

Caused by:
  failed to load source for dependency `candle-core`

Caused by:
  Unable to update https://github.com/huggingface/candle.git?rev=main

Caused by:
  revspec 'main' not found; class=Reference (4); code=NotFound (-3)


In [18]:
# Run this in a Colab cell
!nvcc --version
!nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
Tue Feb  3 23:52:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8       